# Erase — capabilities walkthrough

This notebook runs Bria **Erase** (object removal / inpainting) end to end: install the **`erase`** package from **AWS CodeArtifact**, download the model weights from **Hugging Face**, then remove a masked object from a sample image with the in-process two-stage pipeline (LaMa coarse fill → SDXL ControlNet refine).

**Environment:** an NVIDIA GPU with ≥ 16 GB VRAM, Python **3.10+**. It runs the standard `torch` + `diffusers` stack, so the runtime is portable across recent CUDA versions.

**Prerequisites**

- **`BRIA_API_TOKEN`** — used to obtain a short-lived CodeArtifact credential for the **`bria-erase`** repository.
- **`HF_TOKEN`** — for Hugging Face access to the **`briaai/erase`** model repo.
- Network access to the Bria Engine, AWS CodeArtifact, and Hugging Face.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # loads BRIA_API_TOKEN (and friends) from a local .env file, if present

## 1. CodeArtifact token (Bria Engine)

Request a short-lived credential for the **`bria-erase`** PyPI repository. The Engine expects your Bria API token in the **`api_token`** header (set from `BRIA_API_TOKEN`).

In [ ]:
import os

import requests

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN before running this notebook.")

resp = requests.get(
    "https://engine.prod.bria-api.com/v2/auth/access/code_artifact",
    params={"repository": "bria-erase"},
    headers={"api_token": BRIA_API_TOKEN},
    timeout=60,
)
resp.raise_for_status()
result = resp.json()["result"]
CODE_ARTIFACT_PASSWORD = result["authorization_token"].strip()
print("CodeArtifact credential acquired. Expires:", result.get("expiration"))

## 2. Install `erase` from CodeArtifact

Uses the `authorization_token` from the previous step as the password for the `bria-erase` PyPI simple index (CodeArtifact username `aws`). A **single index** is enough — `bria-erase` has `bria-external` and `bria-diffusers` as upstream repositories, so the Bria packages (`erase`, `bria-external-ml`/`image`, `bria-diffusers`) all resolve through it; `torch`, `diffusers`, etc. come from PyPI (pip's default index). There is no CPU/GPU role split.

In [ ]:
import subprocess
import sys
from urllib.parse import quote

# One index is enough: bria-erase has bria-external + bria-diffusers as upstreams, so pip resolves
# erase and its Bria deps through this single token/index. torch/diffusers/etc. come from PyPI (default).
encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
CA = "https://aws:" + encoded_password + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-erase/simple/"


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(
        install_cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)

    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "erase",
        "--extra-index-url",
        CA,
    ]
)

## 3. Download model weights from Hugging Face

Fetch the weights from **`briaai/erase`** with `HF_TOKEN`, then pass the resulting **local directories** into the pipeline configuration. The library is not opinionated about where weights live — the repo id lives here in the example, not baked into the package.

`allow_patterns` limits the download to what the eraser needs (`vae/`, `controlnet_xl_eraser/`, the `BRIA-2.3-FAST-LORA-FUSED` base + scheduler, and the LaMa `lama/model.pt`).

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN for Hugging Face access to briaai/erase.")

WEIGHTS_REPO = "briaai/erase"
weights = Path(
    snapshot_download(
        repo_id=WEIGHTS_REPO,
        token=hf_token,
        allow_patterns=[
            "vae/**",
            "controlnet_xl_eraser/**",
            "BRIA-2.3-FAST-LORA-FUSED/**",
            "lama/**",
        ],
    )
)
print("weights at", weights)

## 4. Load the sample image + mask

The mask is white (255) over the region to erase. If it does not match the image size it is resized with nearest-neighbour, then cleaned to a hard binary mask.

In [ ]:
import io

import numpy as np
import requests
from IPython.display import display
from PIL import Image

IMG_URL = "https://labs-assets.bria.ai/sandbox-example-inputs/eraser_image_example.jpg"
MASK_URL = "https://labs-assets.bria.ai/sandbox-example-inputs/eraser_mask_example.jpg"


def _download(url):
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()  # clear HTTPError on 404/500 instead of a confusing image-parse error
    return io.BytesIO(resp.content)


image = Image.open(_download(IMG_URL)).convert("RGB")
mask_raw = Image.open(_download(MASK_URL)).convert("L")
if mask_raw.size != image.size:
    mask_raw = mask_raw.resize(image.size, Image.NEAREST)
mask = Image.fromarray(((np.array(mask_raw) > 127) * 255).astype(np.uint8))
print("image", image.size, "| mask", mask.size)
display(image, mask)

## 5. Build configuration and start the pipeline

Pass the downloaded weight directories into `EraseConfig`. The two stage-2 directories (`vae`, `controlnet_xl_eraser`) and the `BRIA-2.3-FAST-LORA-FUSED` base + scheduler are all under the snapshot; `lama/model.pt` is the stage-1 checkpoint.

In [ ]:
import torch

from erase import Erase, EraseConfig, EraseInput

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Erase.")

config = EraseConfig(
    lama_model_path=str(weights / "lama" / "model.pt"),
    vae_model_path=str(weights / "vae"),
    eraser_controlnet_path=str(weights / "controlnet_xl_eraser"),
    base_pipe_model_path=str(weights / "BRIA-2.3-FAST-LORA-FUSED"),
    scheduler_config_path=str(weights / "BRIA-2.3-FAST-LORA-FUSED" / "scheduler" / "scheduler_config.json"),
)
pipeline = Erase(config=config)
pipeline.setup()
print("Pipeline setup complete.")

## 6. Erase the masked object

In [ ]:
import time

Path("outputs").mkdir(exist_ok=True)

t0 = time.time()
result = pipeline.execute(EraseInput(image=image, mask=mask)).image
print(f"erased {result.size} in {time.time() - t0:.1f}s")
result.save("outputs/erased.png")
display(result)

## 7. Side-by-side comparison

input | masked region (highlighted) | erased result.

In [ ]:
inp = np.asarray(image).astype(int)
m = np.asarray(mask) > 0
overlay = inp.copy()
overlay[m] = (inp[m] * 0.35 + np.array([255, 40, 40]) * 0.65).astype(int)

h, w = inp.shape[:2]
combo = Image.new("RGB", (w * 3 + 20, h), "white")
for i, a in enumerate([image, Image.fromarray(overlay.astype(np.uint8)), result.convert("RGB")]):
    combo.paste(a, (i * (w + 10), 0))
combo.save("outputs/compare.png")
display(combo)

## 8. Cleanup

In [ ]:
pipeline.cleanup()
print("Cleanup done.")